# 05. ジェンダー差を最小限の指標で見る

このノートブックは、**01 が作った `data/works/bacteria/paper_authors.parquet`（1行 = 「1人の著者 × 1本の論文」）を読み込むだけ**で、バクテリオファージ（細菌に感染するウイルス）研究における男女の業績・研究環境の差を、できるだけシンプルな指標で観察します。

## この分析が答えたい問い
「女性はデビュー（初めての論文）時に、有力な研究者とつながりやすい」——もしこれが本当だとして、それは **女性という属性そのものの傾向**なのでしょうか？ それとも単に、**女性研究者の参入が最近になって増えた（＝新しい世代に偏っている）**せいで、そう見えているだけなのでしょうか？

このように「見かけの差」が「別の要因」で説明できてしまうことを **交絡（こうらく, confounding）** と呼びます。ここでは **参入年（＝研究を始めた年 = コホート）をそろえて比較する**ことで、この交絡を切り分けます。

## 分析の流れ
1. **著者単位**（§1）: 論文数・キャリア長・FWCI を男女で比較。
2. **出版年ごと**（§2）: 各年の論文数と FWCI を男女で比較。
3. **エントリー年（first_year）ごと**（§3）: 参入時期をそろえて比較。
4. **共著**（§4）: 共著者数とデビュー時の「最強の共著者」を、エントリー年別に。
5. **生存率**（§5）: 参入から何年出版を続けたか（＝途中で辞めていく「漏れるパイプライン」）。
6. **キャリアイヤー別 top-1%**（§6）: 差が「入口（デビュー）」で開くのか、「キャリアが進むにつれて」開くのか。

## 使う指標について（初心者向けメモ）
- **authorship（オーサーシップ）**: 「誰が・どの論文を書いたか」の1組。1人が10本書けば 10 authorships。
- **FWCI（Field-Weighted Citation Impact）**: OpenAlex が計算済みの、**分野と出版年で正規化した被引用インパクト**。**1.0 = 同じ分野・同じ年の世界平均**で、2.0 なら平均の2倍引用されたことを意味します。分野や古さの違いを気にせず論文の注目度を比べられるので、「被引用数は古い論文ほど貯まる」という時間バイアスを避けられます。
- **コホート（cohort）**: ここでは「同じ年に最初の論文を出した著者の集団」のこと。

> **重要な注意**: ここでのジェンダーは `gender-guesser` による**名前ベースの推定**です（unknown が多数・誤分類・文化差・ノンバイナリーな性のあり方が見えなくなる、などの限界があります）。これは記述統計の**例示**であり、能力差や因果関係を主張するものではありません。研究用途では倫理的・方法論的な検証が必須です。


## セットアップ（最初に1回だけ実行）

次のセルは**分析の下準備だけ**を行います。ローカルの Jupyter では、必要なライブラリが入っているかを確認し、作業フォルダを調整します。Google Colab では、ライブラリのインストールとデータの取得（GitHub からのダウンロード）も自動で行います。**中身を細かく理解する必要はありません**——そのまま実行すればOKです。


In [1]:
# === セットアップ（このセルを最初に実行）===
# ローカル(Jupyter): 依存の確認と作業ディレクトリの調整のみ。
# Google Colab   : 依存インストール＋リポジトリ取得も自動で行う。
import os, sys

# データを変更して他ノートと共有・永続化したい人だけ True に（Google Drive をマウントします）。
# False（既定）なら使い捨てランタイム内で完結し、必要データは GitHub Release から取得します（権限プロンプト不要・完全ワンクリック）。
USE_DRIVE = False

if 'google.colab' in sys.modules:
    !pip -q install requests pandas pyarrow numpy matplotlib scikit-learn scipy powerlaw networkx igraph leidenalg umap-learn gender-guesser iso4 nltk openai
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')                      # Trueにした人だけ Drive 権限を承認（各自のDrive・同名でOK）
        BASE = '/content/drive/MyDrive/sciscitutorial'     # ノート間・セッション間でデータを永続共有
    else:
        BASE = '/content/sciscitutorial'                   # 使い捨てランタイム内。権限不要（データはRelease/APIから）
    if not os.path.exists(f'{BASE}/.git'):
        !git clone -q https://github.com/asatani/sciscitutorial.git {BASE}
    os.chdir(BASE)

# code/ から起動した場合はプロジェクトルート（data/ がある場所）へ移動する。
if os.path.basename(os.getcwd()) == 'code':
    os.chdir('..')

# 事前計算済みデータが無ければ GitHub Release から取得し data/ 以下に展開する。
# works=対象トピックの論文, career=03 §3-2 用のランダム著者, supplementary=06(DI)用のエッジリスト。
def ensure_data(name, works=False, career=False, supplementary=False):
    import urllib.request, zipfile
    RELEASE = 'https://github.com/asatani/sciscitutorial/releases/download/data-v1'
    needs = []
    if works:         needs.append((f'data/works/{name}', f'works_{name}.zip'))
    if career:        needs.append(('data/career', 'career.zip'))
    if supplementary: needs.append(('data/supplementary', 'supplementary.zip'))
    os.makedirs('data', exist_ok=True)
    for path, asset in needs:
        if os.path.exists(path):
            continue
        print('downloading', asset, '...')
        zip_path = f'data/{asset}'
        urllib.request.urlretrieve(f'{RELEASE}/{asset}', zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('data')
        os.remove(zip_path)


## 0. 準備 — データを読み込み、名前から性別を推定する

ここからが分析の本体です。まず次のセルで3つのことを行います。

1. **データの読み込み**: `paper_authors.parquet`（1行 = 1著者 × 1論文）を読み込む。
2. **前処理**: 出版年・FWCI を数値に直し、欠損した行を落とす。各論文の**共著者数**を数える。
3. **性別の推定**: 著者名の**ファーストネーム（先頭の単語）**を `gender-guesser` に渡し、`man` / `woman` / `unknown` に振り分ける。


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gender_guesser.detector as gender_guesser

# --- 使うデータセットを選ぶ（'bacteria'=細菌/ファージ T11048 / 'qc'=量子計算 T10020）---
NAME     = 'bacteria'              # ← 'qc' に変えれば量子データセットに切替
DATA_DIR = f'data/works/{NAME}'    # 入力（parquet）のある場所
OUT_DIR  = f'output/{NAME}'        # 図の保存先
ensure_data(NAME, works=True)      # データが手元に無ければ GitHub Release から取得
os.makedirs(OUT_DIR, exist_ok=True)

# グラフの見た目の既定値（枠線を減らす・薄いグリッド・タイトル太字など。分析結果には影響しない）
plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox': 'tight', 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False})
COLOR = {'man': '#4C72B0', 'woman': '#C44E52'}   # 図の色: 男=青 / 女=赤

# --- データ読み込み ---
# 1行 = 「1著者 × 1論文」。同じ論文に3人著者がいれば、その論文は3行に分かれて入っている。
# fwci = その論文の Field-Weighted Citation Impact（分野・年で正規化した被引用インパクト, 1.0=世界平均）。
paper_authors = pd.read_parquet(f'{DATA_DIR}/paper_authors.parquet')

# 文字列で入っている列を数値に変換（変換できない値は NaN=欠損 にする）
paper_authors['publication_year'] = pd.to_numeric(paper_authors['publication_year'], errors='coerce')
paper_authors['fwci'] = pd.to_numeric(paper_authors['fwci'], errors='coerce')
# 出版年か著者IDが欠けている行は分析に使えないので落とす
paper_authors = paper_authors.dropna(subset=['publication_year', 'author_id'])
# 各論文の共著者数 = その論文の著者数 − 1（自分を除く）。work_id ごとの行数から計算する。
paper_authors['n_coauthors'] = paper_authors.groupby('work_id')['author_id'].transform('size') - 1

# --- 名前から性別を推定 ---
detector = gender_guesser.Detector(case_sensitive=False)   # 大文字小文字を区別しない推定器
# ファーストネーム = 著者名の先頭の単語（"J. Smith" → ピリオドを空白にしてから最初のトークン "J"）
first_name = paper_authors['author_name'].fillna('').str.replace('.', ' ', regex=False).str.split().str[0]
# gender-guesser の 6 分類のうち male/female 系を man/woman にまとめ、それ以外(androgyny/unknown)は unknown に
GENDER_MAP = {'male': 'man', 'mostly_male': 'man', 'female': 'woman', 'mostly_female': 'woman'}
paper_authors['gender'] = first_name.apply(detector.get_gender).map(GENDER_MAP).fillna('unknown')

# 全体の規模（authorship数・著者数）と、性別が unknown だった割合を表示
print(f'{len(paper_authors):,} authorships | {paper_authors.author_id.nunique():,} authors | '
      f'unknown gender = {(paper_authors.gender == "unknown").mean():.0%}')


### 読み込み結果

- **386,167 authorships**（著者 × 論文の組）/ 延べ **192,664 人**の著者。
- そのうち **39%** は名前から性別を判定できず `unknown` になりました。ファーストネームがイニシャルだけ（"J."）だったり、辞書に載っていない名前だったりするためです。この 39% は以降の男女比較から自然に外れます（＝男女のどちらかに判定できた人だけを比べます）。


### 著者単位のテーブルを作る

いまのデータは「1行 = 1論文」ですが、ここからは**「1行 = 1人の著者」**の視点で見たいので、著者ごとに集計し直します。各著者について、性別・最初の出版年（`first_year`）・最後の出版年・論文数・平均FWCI・平均共著者数をまとめ、**論文が2本以上（`papers >= 2`）の著者だけ**を残します（1本だけの人は「研究者」としての傾向が読みにくいため）。`authors_mw` は、そのうち男女に判定できた著者だけを取り出した比較用のテーブルです。


In [ ]:
# 著者ごとに1行へ集計する。groupby('author_id') = 同じ著者の行をまとめる。
authors_df = paper_authors.groupby('author_id').agg(
    gender=('gender', lambda s: s.mode().iat[0]),   # その著者で最頻の性別ラベル（論文ごとにブレる場合の代表値）
    first_year=('publication_year', 'min'),         # 最初の出版年 = デビュー年
    last_year=('publication_year', 'max'),          # 最後の出版年
    papers=('work_id', 'nunique'),                  # 執筆した論文数（重複を除いた本数）
    mean_fwci=('fwci', 'mean'),                     # 平均FWCI（§1では使わない: 外れ値に弱いため）
    mean_coauthors=('n_coauthors', 'mean'))         # 1論文あたりの平均共著者数
authors_df['career_len'] = authors_df.last_year - authors_df.first_year + 1   # キャリア長（年数, 最初と最後を含む）
authors_df = authors_df[authors_df.papers >= 2]                    # 論文2本以上の著者だけ残す
authors_mw = authors_df[authors_df.gender.isin(['man', 'woman'])]  # 男女に判定できた著者だけ（比較用）
print('authors kept (papers>=2):', len(authors_df), '| men/women:', authors_mw.gender.value_counts().to_dict())


### 集計結果

論文2本以上の著者は **55,111 人**。うち性別を判定できたのは **男性 22,292 人 / 女性 12,078 人**（残りは unknown）。以降の男女比較は、基本的にこの人たちを対象にします。


## 1. 著者単位で比べる: 論文数・キャリア長・FWCI

1人ずつの著者を、次の4指標で男女比較します。

- **Papers per author**: 1著者あたりの論文数。
- **Career length**: キャリア長（最初と最後の出版年の差 ＋1）。
- **Top-1% FWCI share**: その著者の論文のうち、**全体の上位1%に入る高被引用論文**の割合。
- **Mean log(1+FWCI)**: FWCI の対数平均。

**なぜ平均FWCIを直接使わないのか？** FWCI は、少数の超高被引用論文が飛び抜けて大きい「右に長い裾（右裾, right tail）」を持つ分布です。単純な平均をとると、その一握りの論文に結果が引っ張られてしまいます。そこで、裾に振り回されにくい中心の指標として **対数平均 `mean log(1+FWCI)`** と、裾そのものを見る **top-1% シェア** の2つで「質」を見ます。


In [ ]:
# --- FWCI 由来の2指標を著者テーブルに追加する ---
top1_thr = paper_authors['fwci'].quantile(0.99)     # 「上位1%論文」の FWCI 閾値（全論文の99パーセンタイル）
fw = paper_authors.dropna(subset=['fwci'])          # FWCI がある行だけで計算する
# 著者ごとの log(1+FWCI) 平均。log1p は log(1+x) で、右裾の巨大値の影響を圧縮する。
authors_df['log_fwci'] = np.log1p(fw['fwci']).groupby(fw['author_id']).mean()
# 著者ごとの「上位1%論文の割合」。(fwci >= 閾値) の True/False を著者平均すると割合になる。
authors_df['top1_fwci'] = (fw['fwci'] >= top1_thr).groupby(fw['author_id']).mean()
authors_mw = authors_df[authors_df.gender.isin(['man', 'woman'])]   # 新しい列を足したので比較用テーブルを作り直す

# 表示する4指標（データ列名 → グラフ用の表示名）
metrics = {'papers': 'Papers per author', 'career_len': 'Career length (yrs)',
           'top1_fwci': 'Top-1% FWCI share', 'log_fwci': 'Mean log(1+FWCI)'}
# 男女別に median と mean の表を出す
display(authors_mw.groupby('gender')[list(metrics)].agg(['median', 'mean']).round(3))

# 4指標を棒グラフで男女比較。タイトルに W/M（女性平均 ÷ 男性平均, 1.0=互角）を表示。
fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
for a, (col, title) in zip(ax, metrics.items()):
    m = authors_mw.groupby('gender')[col].mean()
    a.bar(['man', 'woman'], [m['man'], m['woman']], color=[COLOR['man'], COLOR['woman']])
    a.set_title(f'{title}\n(W/M={m["woman"] / m["man"]:.2f})')
fig.suptitle('Author level: men vs women'); fig.tight_layout()
fig.savefig(f'{OUT_DIR}/new_author_level.pdf'); plt.show()


### 結果

| 指標（平均） | 男性 | 女性 | W/M |
|---|---|---|---|
| Papers per author | 5.15 | 4.33 | 0.84 |
| Career length | 9.60 年 | 6.79 年 | 0.71 |
| Top-1% FWCI share | 0.014 | 0.013 | ほぼ同じ |
| Mean log(1+FWCI) | 1.170 | 1.145 | ほぼ同じ |

- **量（論文数・キャリア長）は男性が上**: 女性は男性の 0.84 倍の論文数、0.71 倍のキャリア長。
- **質（FWCI 指標）はほぼ互角**: 対数平均も top-1% シェアも、男女でほとんど差がありません。

つまり著者単位で見ると、差は「どれだけ長く・多く出版したか」に集中していて、「1本1本のインパクト」には表れていません。ただしこの比較は**参入時期の違いを無視している**点に注意——女性の参入が最近に偏っていれば、キャリアが短く見えるのは当然です。これを次から切り分けます。


## 2. 出版年ごとに比べる

こんどは**出版年（論文が出た年）**を横軸にして、各年ごとに男女を比較します。

- **Authorships per year**: その年の authorship 数（男女それぞれの「出版の量」）。
- **Top-1% FWCI share (%)**: その年の論文が上位1%に入る割合。
- **Mean log(1+FWCI)**: その年の論文の対数平均FWCI。

これで「近年になって女性の出版が増えたか」「各年の論文の質に男女差があるか」が見えます。


In [ ]:
# 男女のみを対象に、(出版年 × 性別) で集計する。unstack('gender') で列を man/woman に展開。
gp = paper_authors[paper_authors.gender.isin(['man', 'woman'])].copy()
papers_y = gp.groupby(['publication_year', 'gender'])['work_id'].size().unstack('gender')   # 各年の authorship 数
gf = gp.dropna(subset=['fwci'])                            # FWCI 指標は fwci のある行だけで計算
# 各年の「上位1%論文の割合(%)」
top_y = (gf['fwci'] >= top1_thr).groupby([gf['publication_year'], gf['gender']]).mean().unstack('gender') * 100
# 各年の対数平均FWCI
log_y = np.log1p(gf['fwci']).groupby([gf['publication_year'], gf['gender']]).mean().unstack('gender')
yrs = papers_y.index[papers_y.index >= 1990]              # 1990年以降だけ描画（それ以前はデータが薄い）

# 3枚並べて折れ線で男女比較
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, (data, title) in zip(ax, [(papers_y, 'Authorships per year'),
                                 (top_y, 'Top-1% FWCI share (%)'),
                                 (log_y, 'Mean log(1+FWCI)')]):
    for who in ['man', 'woman']:
        a.plot(yrs, data.loc[yrs, who], color=COLOR[who], lw=2, label=who)
    a.set_title(title); a.set_xlabel('publication year'); a.legend()
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/new_by_pubyear.pdf'); plt.show()


### 結果

- **出版量**: どの年も男性の authorship が女性より多いものの、**近年に向けて男女とも急増**し、特に女性の伸びが目立ちます（＝女性の参入が最近に偏っている証拠）。
- **質（top-1% シェア・対数平均FWCI）**: 男女の線は**ほぼ重なっています**。各年の論文の質に、はっきりした男女差はありません。

量には差があっても、その年その年の「1本あたりの質」は男女でほぼ同じ、というのがここでの読み取りです。


## 3. エントリー年（研究を始めた年）でそろえて比べる

ここが**交絡を切り分ける核心**です。§1・§2 で見た差は「女性の参入が最近に偏っている」せいかもしれません。そこで、横軸を **エントリー年（`first_year` = その著者が最初に論文を出した年 = コホート）** にして、**同じ年にデビューした男女どうし**で比べます。

- **Mean papers per author**: そのコホートの1著者あたり平均論文数。
- **Top-1% FWCI share (%)** と **Mean log(1+FWCI)**: 質指標（著者単位）。

直近の年は「まだ追跡できる期間が短い」ので、集計から除外します（最後の2年ぶんを除く）。


In [ ]:
# 対象コホート: 1995年 〜 (最新年 − 1)。直近2年ほどは追跡期間が短すぎるので除く。
yrs = range(1995, int(paper_authors.publication_year.max()) - 1)
# 同じ (エントリー年 × 性別) の著者をまとめる
coh = authors_mw[authors_mw.first_year.isin(yrs)].groupby(['first_year', 'gender'])
papers_c = coh['papers'].mean().unstack('gender')          # コホート別・平均論文数
top_c = coh['top1_fwci'].mean().unstack('gender') * 100    # コホート別・top-1%シェア(%)
log_c = coh['log_fwci'].mean().unstack('gender')           # コホート別・対数平均FWCI

# エントリー年を横軸に、男女を折れ線（点付き）で比較
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, (data, title) in zip(ax, [(papers_c, 'Mean papers per author'),
                                 (top_c, 'Top-1% FWCI share (%)'),
                                 (log_c, 'Mean log(1+FWCI)')]):
    for who in ['man', 'woman']:
        a.plot(data.index, data[who], color=COLOR[who], lw=2, marker='o', ms=3, label=who)
    a.set_title(title); a.set_xlabel('entry year (first publication)'); a.legend()
fig.suptitle('By entry cohort: men vs women'); fig.tight_layout()
fig.savefig(f'{OUT_DIR}/new_by_cohort.pdf'); plt.show()


### 結果

参入年をそろえると、景色が変わります。

- **平均論文数**: 同じコホートで見ると男女の差はかなり縮まります（§1で見えた大きな差は、女性が新しいコホートに多いことの影響が大きかった）。
- **質指標（top-1% ・対数平均FWCI）**: 同年に参入した男女では**同等**で、**年によってはむしろ女性が上回ります**。

つまり §1 で見えた累積的な差の多くは、**参入時期やキャリアの長さという交絡**で説明できる、というのがここでの結論です。


## 4. 共著関係: デビュー時に「有力な人」とつながっているか

冒頭の問い——「女性はデビュー時に有力者とつながりやすい」——を、コホートをそろえて検証します。ここで見るのは2つ。

- **Mean co-authors per paper**: 1論文あたりの平均共著者数。
- **Debut: strongest co-author (#papers)**: `debut_coauthor_papers`。**デビュー論文の共著者のうち、生涯論文数が最も多い人（＝最も多産な共著者）の論文数**。デビュー時にどれだけ「大物」と組んでいたかの代理指標です。

これらを**エントリー年別**に男女で比べ、「有力者とのつながり」が単に**参入の遅さ**で説明できてしまうのかを確かめます。


In [ ]:
# --- 各著者の「デビュー論文の最強共著者」を求める ---
papers_all = paper_authors.groupby('author_id')['work_id'].nunique()          # 著者ごとの生涯論文数（=多産さ）
work_authors = paper_authors.groupby('work_id')['author_id'].apply(list)      # 論文ごとの著者IDリスト
# デビュー論文 = その著者の最初の出版年に出た論文（min の年に一致する行）
debut = paper_authors[paper_authors.publication_year ==
                      paper_authors.groupby('author_id')['publication_year'].transform('min')]
e = debut[['author_id', 'work_id']].copy()
e['co'] = e['work_id'].map(work_authors)          # デビュー論文の共著者リストを貼り付け
e = e.explode('co'); e = e[e.author_id != e.co]   # 1行1共著者に展開し、自分自身は除く
e['co_papers'] = e['co'].map(papers_all)          # 各共著者の生涯論文数
# 著者ごとに「最も多産な共著者の論文数」= デビュー時の最強共著者
authors_df['debut_coauthor_papers'] = e.groupby('author_id')['co_papers'].max()

# 男女 & 対象コホート(yrs, §3で定義)に絞り、(エントリー年 × 性別)で平均する
authors_mw2 = authors_df[authors_df.gender.isin(['man', 'woman']) & authors_df.first_year.isin(yrs)]
co_c = authors_mw2.groupby(['first_year', 'gender'])['mean_coauthors'].mean().unstack('gender')          # 平均共著者数
ds_c = authors_mw2.groupby(['first_year', 'gender'])['debut_coauthor_papers'].mean().unstack('gender')   # 最強共著者

# 左=平均共著者数 / 右=デビュー時の最強共著者 を、エントリー年別に男女比較
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for who in ['man', 'woman']:
    ax[0].plot(co_c.index, co_c[who], color=COLOR[who], lw=2, marker='o', ms=3, label=who)
    ax[1].plot(ds_c.index, ds_c[who], color=COLOR[who], lw=2, marker='o', ms=3, label=who)
ax[0].set_title('Mean co-authors per paper')
ax[1].set_title('Debut: strongest co-author (#papers)')
for a in ax:
    a.set_xlabel('entry year (first publication)'); a.legend()
fig.suptitle('Co-authorship by entry cohort: men vs women'); fig.tight_layout()
fig.savefig(f'{OUT_DIR}/new_coauthorship.pdf'); plt.show()


### 結果

- **共著者数**も **デビュー時の最強共著者**も、**どのエントリー年でも女性が男性と同等〜やや上**でした。

同じ年にデビューした男女で比べても女性のほうが有力者と組んでいる、ということは、女性の「早期のコネクション優位」は**参入時期の遅さでは説明できない**という意味です。冒頭の問いへの答えは「参入が遅いせいではなく、本当に早い段階から有力者とつながっている」となります。


## 5. 研究者としての「生存率」（出版を続けられたか）

参入の遅さ（§3）とは別の角度として、**継続率＝どれだけ長く出版を続けたか**を見ます。生存率 **S(T) = 参入から T 年後もまだ出版している人の割合**（＝生存曲線）です。

ここで重要なのが**右側打ち切り（right-censoring）の補正**です。最近デビューした人は、まだ T 年ぶんの時間が経っていないので「T年後も活動中か」を判定できません。そこで各 T では、**少なくとも T 年は追跡できた著者だけ**を分母にします。この節は全著者（papers≥1）が対象です。

- **左**: 生存曲線 S(T)（男 vs 女）。W/M が時間とともに下がれば「**漏れるパイプライン**（女性ほど途中で抜けていく）」の兆候。
- **右**: 5年生存率をエントリー年別に（最近参入の影響を除いても、女性が一貫して低いか）。

> 注意: これは細菌/ファージという**トピックのコーパス**なので、ここでの「生存」＝**この分野で出版し続けたか**です。分野を移った人も「脱落」に見えるため、真の学界離脱より多めに出ます。男女で分野転向の傾向が同じなら、相対比較（W/M）は有効です。


In [ ]:
# この節は全著者(papers>=1)が対象。著者ごとに 性別・デビュー年・最終出版年 を集計する。
sv = paper_authors.groupby('author_id').agg(
    gender=('gender', lambda s: s.mode().iat[0]),
    first_year=('publication_year', 'min'),
    last_year=('publication_year', 'max'))
sv['span'] = sv.last_year - sv.first_year          # span = 活動年数（最初の出版から最後の出版まで）
sv = sv[sv.gender.isin(['man', 'woman'])]
YMAX = int(paper_authors.publication_year.max())   # データ内の最新年

# S(T) = P(span >= T)。打ち切り補正: 各 T では「T年前までにデビュー済み(=T年追跡できた)」著者だけを分母にする。
T = np.arange(1, 16)
surv = {who: [(sv[(sv.gender == who) & (sv.first_year <= YMAX - t)].span >= t).mean() for t in T]
        for who in ['man', 'woman']}
# 5年生存率をエントリー年別に（span>=5 を True/False にしてコホート平均）
yrs = range(1995, YMAX - 4)
s5 = (sv[sv.first_year.isin(yrs)].assign(s=lambda d: d.span >= 5)
      .groupby(['first_year', 'gender'])['s'].mean().unstack('gender'))

# 左=生存曲線 S(T) / 右=5年生存率のコホート推移
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for who in ['man', 'woman']:
    ax[0].plot(T, surv[who], color=COLOR[who], lw=2, marker='o', ms=3, label=who)
    ax[1].plot(s5.index, s5[who], color=COLOR[who], lw=2, marker='o', ms=3, label=who)
ax[0].set_title('Survival S(T): still publishing T years after entry')
ax[0].set_xlabel('years since first publication'); ax[0].set_ylabel('share still active'); ax[0].legend()
ax[1].set_title('5-year survival by entry cohort')
ax[1].set_xlabel('entry year (first publication)'); ax[1].set_ylabel('P(active >= 5 yrs)'); ax[1].legend()
fig.suptitle('Academic survival within the topic: men vs women'); fig.tight_layout()
fig.savefig(f'{OUT_DIR}/new_survival.pdf'); plt.show()

# 代表的な T について、男女の生存率と W/M を数値で表示
print('survival S(T)  (men, women, W/M):')
for t in [1, 5, 10, 15]:
    e = sv[sv.first_year <= YMAX - t]                          # T年追跡できた著者だけ
    m = (e[e.gender == 'man'].span >= t).mean(); w = (e[e.gender == 'woman'].span >= t).mean()
    print(f'  T={t:2d}  {m:.3f}  {w:.3f}  W/M={w/m:.2f}')


### 結果

数値出力（参入から T 年後もまだ出版している割合）:

| T | 男性 | 女性 | W/M |
|---|---|---|---|
| 1年 | 0.312 | 0.266 | 0.85 |
| 5年 | 0.223 | 0.162 | 0.73 |
| 10年 | 0.175 | 0.118 | 0.67 |
| 15年 | 0.144 | 0.094 | 0.65 |

- **W/M が時間とともに低下**（0.85 → 0.65）。つまり時間が経つほど、女性の残存率が男性に対して下がっていく＝**漏れるパイプライン**の兆候です。
- 右図の5年生存率をエントリー年別に見ても、女性が一貫して低め。最近参入の影響を除いても差は残ります。

質（§1〜§3）ではほぼ互角なのに、**続けられるかどうか（定着）には男女差がある**——これがこの節の要点です。


## 6. キャリアイヤー別に見る: top-1% の差はどこで開くのか

最後に、「上位1%論文に入る割合の男女差」が **入口（デビュー）で開くのか、それともキャリアが進むにつれて開くのか**を分けて見ます。

各論文を「その著者の**何年目**に出したか（`pub_year − first_year` = キャリアイヤー）」でグループ分けし、`0 / 1-2 / 3-5 / 6-10 / 11-20` 年目のビンごとに、**その論文が top-1% FWCI に入る割合**を男女で比較します（エラーバーは二項分布の 95% 信頼区間）。


In [ ]:
# 男女 & FWCI のある論文だけを対象に、各論文へ「著者の何年目か」を付ける。
gp = paper_authors[paper_authors.gender.isin(['man', 'woman'])].dropna(subset=['fwci']).copy()
gp['first_year'] = gp.groupby('author_id')['publication_year'].transform('min')   # 著者のデビュー年
gp['career_year'] = (gp.publication_year - gp.first_year).astype(int)             # キャリアイヤー(0=デビュー年)
gp['top1'] = gp.fwci >= top1_thr                                   # その論文が上位1%か（top1_thr は §1 で定義）
labels = ['0', '1-2', '3-5', '6-10', '11-20']
# キャリアイヤーを5つのビンに区切る（境界を .5 にして整数を確実に振り分ける）
gp['career_bin'] = pd.cut(gp.career_year, [-0.5, 0.5, 2.5, 5.5, 10.5, 20.5], labels=labels)

# ビンごとの top-1% 割合(%)を、二項分布の95%信頼区間つきで男女比較
fig, ax = plt.subplots(figsize=(7, 4.2))
for who in ['man', 'woman']:
    grp = gp[gp.gender == who].groupby('career_bin')['top1']
    pct = grp.mean() * 100                                        # ビンごとの top-1% 割合(%)
    ci = np.sqrt((pct / 100) * (1 - pct / 100) / grp.size()) * 100 * 1.96   # 二項95%CI = 1.96*sqrt(p(1-p)/n)
    ax.errorbar(range(len(labels)), pct.values, yerr=ci.values,
                color=COLOR[who], lw=2, marker='o', capsize=3, label=who)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_xlabel('career year at publication (pub_year - first_year)')
ax.set_ylabel('Top-1% FWCI share (%)'); ax.set_title('Top-1% FWCI share vs career year (95% CI)')
ax.legend(); fig.tight_layout(); fig.savefig(f'{OUT_DIR}/new_top1_vs_careeryear.pdf'); plt.show()


### 結果

- **デビュー（0年目）は男女ほぼ同じ**（およそ 0.9%）。スタート地点に差はありません。
- ところが、**男性はその後 〜1.5% まで上がって高止まり**、**女性は伸びずに 〜0.7% へと下がって**いきます。

つまり top-1% の男女差は「**入口**」ではなく、「**キャリアが進むにつれて**」開いていきます。これは、生存バイアス（続けられた人だけが残る）や、成功が次の成功を呼ぶ**累積優位（マタイ効果 / マチルダ効果）**と整合的です。§5 の「漏れるパイプライン」とあわせると、差が生まれるのは入口よりも**その後の環境・定着の側**にある、と読めます。


## まとめ

- **論文数・キャリア長は男性が上**だが、**1本あたりの質（FWCI）は男女ほぼ互角**（§1）。
- **出版年・エントリー年をそろえる**と、質は同等〜むしろ女性が優位の年もある（§2, §3）。→ §1で見えた累積の差の多くは、**参入時期・在籍長という交絡**で説明できる。
- 女性の**デビュー時の有力者とのつながりは、参入の遅さでは説明できない**（同じコホートでも女性が同等〜上, §4）。
- ただし**継続率（生存）は女性が低く、時間とともに差が開く**（§5）。top-1% の差も**入口ではなく、キャリアの進行とともに開く**（§6）。→ 問題は「入口の実力」より「その後も続けられる環境」の側にある可能性。

**留保**: 性別は名前ベースの推定（unknown 多数）であること、papers≥2 などの標本の選び方、そして**これは記述統計であって因果ではない**こと。さらにトピック限定コーパスなので「生存」は分野内での継続を指すこと。これらを踏まえた上での観察です。
